# Random Forest Final

I prepared an RF-only Kaggle candidate using the settings and training sizes selected in V5. I used 400 image examples and 800 sentiment examples, not all labeled data. The 800-example sentiment fit took about 76 seconds in V5, and larger refits have not been tested.

First, I imported the libraries and loaded the saved V5 results.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
import platform
import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data_loader import load_task, split_train_val
from src.models.random_forest import RandomForest

data_dir = project_root.parent / "data"
result_dir = project_root / "results/random_forest"
v5 = pd.read_csv(result_dir / "random_forest_v5_summary.csv", float_precision="round_trip")
summary_rows = []
prediction_frames = []
print(f"Python {platform.python_version()}, NumPy {np.__version__}, pandas {pd.__version__}")


Python 3.14.3, NumPy 2.4.3, pandas 3.0.1


I refitted the image forest on the same selected training subset and predicted the test examples.

In [ ]:
task = "image"
best = v5.loc[v5.dataset == task].sort_values(["validation_accuracy", "training_samples"], ascending=[False, True]).iloc[0]
training_samples = int(best.training_samples)
assert training_samples == 400
X, y, X_test, test_ids = load_task(task, data_dir=data_dir)
X_train_full, X_val, y_train_full, y_val = split_train_val(X, y, val_fraction=0.2, seed=42)
X_train, y_train = X_train_full[:training_samples], y_train_full[:training_samples]
assert X_train.shape == (training_samples, 500)
assert X_test.shape == (5000, 500)
clf = RandomForest(n_trees=int(best.n_trees), max_depth=int(best.max_depth), min_samples=1, n_features="sqrt", seed=42)
assert clf.n_trees == 20 and clf.max_depth == 8
start = perf_counter()
clf.fit(X_train, y_train)
fit_time = perf_counter() - start
# Check refit consistency; this is the existing V5 holdout, not a new evaluation set.
v5_predictions = pd.read_csv(result_dir / f"random_forest_v5_{task}_validation.csv")
assert v5_predictions.actual_label.tolist() == y_val.tolist()
assert clf.predict(X_val) == v5_predictions.predicted_label.tolist()
start = perf_counter()
test_predictions = clf.predict(X_test)
prediction_time = perf_counter() - start
assert len(test_predictions) == len(test_ids) == 5000
assert all(0 <= p <= 49 for p in test_predictions)
frame = pd.DataFrame({"id": test_ids, "label": test_predictions})
assert frame.id.is_unique and not frame.isna().any().any()
prediction_frames.append(frame)
summary_rows.append({"dataset": task, "version": "Final", "n_trees": clf.n_trees,
    "max_depth": clf.max_depth, "min_samples": clf.min_samples, "n_features": clf.n_features,
    "criterion": "entropy", "normalized": False, "seed": 42,
    "available_labeled_samples": len(y), "final_training_samples": training_samples,
    "training_subset": "First N rows of the seed-42 stratified 80% training split, matching V5",
    "selected_using": "V5 full held-out validation; ties prefer smaller training size",
    "test_samples": len(test_ids), "fit_time_seconds": fit_time,
    "test_prediction_time_seconds": prediction_time})
print(f"{task}: refitted {training_samples}/{len(y)} labeled examples: V5 predictions reproduced exactly.")
print(f"Fit={fit_time:.9f} s, 5000 test predictions={prediction_time:.9f} s")


image: refitted 400/20000 labeled examples; V5 predictions reproduced exactly.
Fit=2.964845041 s, 5000 test predictions=0.079986667 s


Then I refitted the sentiment forest on its selected subset and predicted its test examples.

In [ ]:
task = "sentiment"
best = v5.loc[v5.dataset == task].sort_values(["validation_accuracy", "training_samples"], ascending=[False, True]).iloc[0]
training_samples = int(best.training_samples)
assert training_samples == 800
X, y, X_test, test_ids = load_task(task, data_dir=data_dir)
X_train_full, X_val, y_train_full, y_val = split_train_val(X, y, val_fraction=0.2, seed=42)
X_train, y_train = X_train_full[:training_samples], y_train_full[:training_samples]
assert X_train.shape == (training_samples, 768)
assert X_test.shape == (5000, 768)
clf = RandomForest(n_trees=int(best.n_trees), max_depth=int(best.max_depth), min_samples=1, n_features="sqrt", seed=42)
assert clf.n_trees == 20 and clf.max_depth == 8
start = perf_counter()
clf.fit(X_train, y_train)
fit_time = perf_counter() - start
# Check refit consistency; this is the existing V5 holdout, not a new evaluation set.
v5_predictions = pd.read_csv(result_dir / f"random_forest_v5_{task}_validation.csv")
assert v5_predictions.actual_label.tolist() == y_val.tolist()
assert clf.predict(X_val) == v5_predictions.predicted_label.tolist()
start = perf_counter()
test_predictions = clf.predict(X_test)
prediction_time = perf_counter() - start
assert len(test_predictions) == len(test_ids) == 5000
assert all(0 <= p <= 2 for p in test_predictions)
frame = pd.DataFrame({"id": test_ids, "label": test_predictions})
assert frame.id.is_unique and not frame.isna().any().any()
prediction_frames.append(frame)
summary_rows.append({"dataset": task, "version": "Final", "n_trees": clf.n_trees,
    "max_depth": clf.max_depth, "min_samples": clf.min_samples, "n_features": clf.n_features,
    "criterion": "entropy", "normalized": False, "seed": 42,
    "available_labeled_samples": len(y), "final_training_samples": training_samples,
    "training_subset": "First N rows of the seed-42 stratified 80% training split, matching V5",
    "selected_using": "V5 full held-out validation; ties prefer smaller training size",
    "test_samples": len(test_ids), "fit_time_seconds": fit_time,
    "test_prediction_time_seconds": prediction_time})
print(f"{task}: refitted {training_samples}/{len(y)} labeled examples; V5 predictions reproduced exactly.")
print(f"Fit={fit_time:.9f} s, 5000 test predictions={prediction_time:.9f} s")


sentiment: refitted 800/12000 labeled examples; V5 predictions reproduced exactly.
Fit=75.621720292 s, 5000 test predictions=0.078832875 s


Finally, I matched the predictions to the sample submission by ID.

In [ ]:
sample = pd.read_csv(data_dir / "sample_submission.csv")
assert len(sample) == 10000 and sample.id.is_unique and sample.id.notna().all()
predictions = pd.concat(prediction_frames, ignore_index=True)
assert len(predictions) == 10000 and predictions.id.is_unique
assert set(predictions.id) == set(sample.id)
submission = sample[["id"]].merge(predictions, on="id", how="left", validate="one_to_one", sort=False)
assert submission.columns.tolist() == ["id", "label"]
assert submission.id.tolist() == sample.id.tolist()
assert not submission.isna().any().any()
assert np.equal(submission.label, np.floor(submission.label)).all()
submission["label"] = submission.label.astype(int)
image_mask = submission.id.str.startswith("img_")
text_mask = submission.id.str.startswith("text_")
assert image_mask.sum() == text_mask.sum() == 5000
assert submission.loc[image_mask, "label"].between(0, 49).all()
assert submission.loc[text_mask, "label"].between(0, 2).all()
submission_dir = project_root / "submission/random_forest"
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "submission_final_random_forest.csv"
submission.to_csv(submission_path, index=False)
saved = pd.read_csv(submission_path)
pd.testing.assert_frame_equal(saved, submission)
summary = pd.DataFrame(summary_rows)
assert len(summary) == 2
summary.to_csv(result_dir / "random_forest_final_summary.csv", index=False)
print(summary.to_string(index=False))
print("Verified 10,000 unique IDs in sample-submission order, 5,000 per task, with valid integer labels.")
tracker_path = result_dir / "kaggle_scores.csv"
tracker = pd.read_csv(tracker_path, dtype=str, keep_default_na=False)
assert tracker.version.is_unique
previous_rows = tracker.loc[tracker.version != "Final"].to_dict("records")
final_row = {"version": "Final", "image_validation_accuracy": "", "sentiment_validation_accuracy": "",
    "kaggle_public_score": "", "submitted_to_kaggle": "False",
    "submission_filename": submission_path.name,
    "notes": "RF-only local candidate; 20 trees, depth=8, sqrt, entropy, min_samples=1, seed=42; original features; 400 image/800 sentiment training examples from V5 subsets, NOT full-data training. Historical V5 validation: image 0.0445 on 4000; sentiment 1195/2400. Exact V5 holdout predictions reproduced; no new validation or Kaggle score."}
matching = tracker.version == "Final"
if matching.any():
    for column in ("submission_filename", "notes"):
        tracker.loc[matching, column] = final_row[column]
else:
    tracker = pd.concat([tracker, pd.DataFrame([final_row])], ignore_index=True)
assert tracker.version.is_unique
assert tracker.loc[tracker.version != "Final"].to_dict("records") == previous_rows
tracker.to_csv(tracker_path, index=False)


  dataset version  n_trees  max_depth  min_samples n_features criterion  normalized  seed  available_labeled_samples  final_training_samples                                                        training_subset                                                 selected_using  test_samples  fit_time_seconds  test_prediction_time_seconds
    image   Final       20          8            1       sqrt   entropy       False    42                      20000                     400 First N rows of the seed-42 stratified 80% training split, matching V5 V5 full held-out validation; ties prefer smaller training size          5000          2.964845                      0.079987
sentiment   Final       20          8            1       sqrt   entropy       False    42                      12000                     800 First N rows of the seed-42 stratified 80% training split, matching V5 V5 full held-out validation; ties prefer smaller training size          5000         75.621720                    